# Style2Fit — Training Pipeline
AIPI 540 Mini Hackathon #3  
Run on Google Colab with A100 GPU runtime.  
Project files are in Google Drive: `MyDrive/GAI_hackathon/Style2Fit/`

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/GAI_hackathon/Style2Fit"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Part 0 — Install dependencies
!pip install -q transformers==4.46.0 datasets peft bitsandbytes accelerate trl

In [ ]:
# Check GPU type
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Part 1 — Load and format training data
import json
from datasets import Dataset

# style2fit_training_data.jsonl should be in the project directory on Drive
with open("style2fit_training_data.jsonl", "r") as f:
    raw_data = [json.loads(line) for line in f]

def format_sample(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n{sample['output']}"
    }

formatted = [format_sample(s) for s in raw_data]
dataset = Dataset.from_list(formatted)

print(f"Loaded {len(dataset)} training samples")
print("=" * 60)
print("First example preview:")
print(dataset[0]["text"][:500])

In [ ]:
# Part 2 — Load base model with QLoRA (4-bit quantization)

# === Authentication ===
# Option A: Use Colab Secrets (recommended)
#   Left sidebar → key icon → add HF_TOKEN → toggle "Notebook access" on
# Option B: Paste token directly
#   from huggingface_hub import login
#   login(token="hf_YOUR_TOKEN_HERE")

from huggingface_hub import login
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in via Colab Secrets")
except Exception:
    print("HF_TOKEN not found in Colab Secrets, trying interactive login...")
    login()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ====================================================================
# CHOOSE YOUR MODEL — uncomment ONE of the following:
# ====================================================================
# Option 1: Llama 3.1 8B (requires Meta approval on HuggingFace)
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Option 2: Mistral 7B (no gated access required — use this to get started)
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
# ====================================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Loaded {MODEL_ID}")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Part 3 — Define demo prompts and generate BEFORE fine-tuning

DEMO_PROMPTS = [
    {"id": "coffee_date_rainy", "gender": "female", "season": "fall",
     "style": "clean girl",
     "prompt": "Coffee date tomorrow morning, it's going to be rainy and chilly around 55\u00b0F."},
    {"id": "tech_internship", "gender": "male", "season": "spring",
     "style": "smart casual",
     "prompt": "First day at my tech internship, business casual dress code. I'm nervous."},
    {"id": "indie_concert", "gender": "neutral", "season": "summer",
     "style": "indie grunge",
     "prompt": "Going to an indie rock concert Saturday night, want to look cool but stand for hours."},
    {"id": "work_presentation", "gender": "female", "season": "winter",
     "style": "power professional",
     "prompt": "Big presentation at work Thursday. Want to look confident but not intimidating."},
    {"id": "lisbon_trip", "gender": "male", "season": "spring",
     "style": "Mediterranean casual",
     "prompt": "Weekend trip to Lisbon in April. Need outfits for sightseeing and dinners."},
    {"id": "summer_wedding", "gender": "female", "season": "summer",
     "style": "garden party",
     "prompt": "Outdoor summer wedding guest, festive attire, 85\u00b0F and sunny."},
    {"id": "rooftop_date", "gender": "male", "season": "fall",
     "style": "urban cool",
     "prompt": "First date at a trendy rooftop bar, Friday evening in October."},
    {"id": "creative_interview", "gender": "neutral", "season": "winter",
     "style": "creative professional",
     "prompt": "Job interview at a creative agency, casual culture but I need to impress."},
]

INSTRUCTION = "You are Style2Fit, a fashion styling assistant. Given a situation described in natural language, generate a complete, structured outfit recommendation. Include specific items with colors and materials, an overall aesthetic label, and a brief explanation of why the outfit works for the situation."

def generate_outfit(prompt_text, model, tokenizer):
    full_prompt = f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{prompt_text}\n\n### Response:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Generate BEFORE fine-tuning
print("Generating BEFORE fine-tuning outputs...")
print("=" * 60)
before_results = {}
for dp in DEMO_PROMPTS:
    print(f"\n>>> {dp['id']}")
    result = generate_outfit(dp["prompt"], model, tokenizer)
    before_results[dp["id"]] = result
    print(result[:200])
    print("---")

import json
with open("before_results.json", "w") as f:
    json.dump(before_results, f, indent=2)
print("\nSaved before_results.json")

In [ ]:
# Part 4 — QLoRA Fine-Tuning
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_config = SFTConfig(
    output_dir="./style2fit-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=512,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_config,
)

print("Starting training...")
trainer.train()

# Save final model
model.save_pretrained("./style2fit-lora-final")
tokenizer.save_pretrained("./style2fit-lora-final")
print("Model saved to ./style2fit-lora-final")

In [ ]:
# Part 5 — Generate AFTER fine-tuning
# The LoRA weights are already applied to the model from Part 4.
# If you need to reload after restart, uncomment below:
# from peft import PeftModel
# model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
# model = PeftModel.from_pretrained(model, "./style2fit-lora-final")

print("Generating AFTER fine-tuning outputs...")
print("=" * 60)
after_results = {}
for dp in DEMO_PROMPTS:
    print(f"\n>>> {dp['id']}")
    result = generate_outfit(dp["prompt"], model, tokenizer)
    after_results[dp["id"]] = result
    print(result[:200])
    print("---")

with open("after_results.json", "w") as f:
    json.dump(after_results, f, indent=2)
print("\nSaved after_results.json")

# Side-by-side comparison
print("\n" + "=" * 80)
print("BEFORE vs AFTER COMPARISON")
print("=" * 80)
for dp in DEMO_PROMPTS:
    print(f"\n{'='*60}")
    print(f"PROMPT: {dp['prompt']}")
    print(f"\n--- BEFORE ---")
    print(before_results[dp["id"]][:300])
    print(f"\n--- AFTER ---")
    print(after_results[dp["id"]][:300])

In [ ]:
# Part 5b — Export combined data for the image generation notebook
# Saved to Drive so notebook 2 can access it directly
demo_data_text = []
for dp in DEMO_PROMPTS:
    demo_data_text.append({
        "id": dp["id"],
        "gender": dp["gender"],
        "season": dp["season"],
        "style": dp["style"],
        "prompt": dp["prompt"],
        "before_text": before_results[dp["id"]],
        "after_text": after_results[dp["id"]],
    })

with open("demo_data_text.json", "w") as f:
    json.dump(demo_data_text, f, indent=2)

print("Saved demo_data_text.json to Drive")
print("This file will be accessible from the image generation notebook via the same Drive path.")